<a href="https://colab.research.google.com/github/laramalkawi81-ops/DS230-Instacart-Project/blob/main/Copy_of_08_explainability_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Explainability & Robustness Tests

This notebook focuses on interpreting the top-performing models
using SHAP and testing model robustness under noise, outliers,
and reduced training data scenarios.


In [ ]:
import pandas as pd
import numpy as np

import shap

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns


Download the best model + Sample Data

In [ ]:

X_test = pd.read_csv(f"{DATA_PATH}/sample_X_test.csv")
y_test = pd.read_csv(f"{DATA_PATH}/sample_y_test.csv")

import joblib
rf_model = joblib.load(f"{DATA_PATH}/rf_model_sample.pkl")


SHAP Explanation

In [ ]:
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values[1], X_test)  # class 1 = reordered


SHAP summary plot shows the most important features affecting
whether a product is reordered.


Robustness Tests – Noise

In [ ]:

for sigma in [0.01, 0.05, 0.1]:
    X_noisy = X_test + np.random.normal(0, sigma*X_test.std(), X_test.shape)
    y_pred = rf_model.predict(X_noisy)
    print(f"Sigma={sigma} -> Accuracy:", accuracy_score(y_test, y_pred))


- Add small random noise to numeric features.
- Measure how accuracy, F1-score, and AUC degrade.
- Typical sigma values tested: 0.01, 0.05, 0.1 × standard deviation.


Robustness Tests – Outliers

In [ ]:
X_outliers = X_test.copy()
n_outliers = int(0.01 * len(X_outliers))
outlier_idx = np.random.choice(X_outliers.index, n_outliers, replace=False)
X_outliers.loc[outlier_idx, 'user_product_count'] *= 10  # outlier
y_pred = rf_model.predict(X_outliers)
print("Accuracy with 1% outliers:", accuracy_score(y_test, y_pred))


- Randomly amplify 1% of feature values to simulate extreme cases.
- Observe model prediction changes.
- Helps identify sensitivity to unusual data points.


Robustness Tests – Reduced Training Data

In [ ]:

X_train_small = X_train.sample(frac=0.3, random_state=42)
y_train_small = y_train.loc[X_train_small.index]

rf_small = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_small.fit(X_train_small, y_train_small)

y_pred_small = rf_small.predict(X_test)
print("Accuracy with 30% training data:", accuracy_score(y_test, y_pred_small))



- SHAP explains the contribution of top features to model predictions.
- Robustness tests show model stability under Gaussian noise,
  injected outliers, and reduced training data.
- These analyses provide confidence in model reliability
  before production deployment.
